# 10. Probar el aplicativo en Colab

Ejecuta **Preparar la prueba** una vez. Después escribe un texto en
**Clasificar un texto**, elige **Solo M2** o **Comparar M0, M1 y M2** y
ejecuta esa celda. Puedes repetirla con los textos que quieras.

La primera prueba descarga los pesos que necesite. Usa CPU y guarda los
archivos en la sesión de Colab. Esta prueba usa los pesos publicados de
`corrida_03_epocas` y muestra polaridad textual y probabilidades. El
entrenamiento y las métricas de evaluación se consultan en los cuadernos 06–09.

In [ ]:
# @title 1. Preparar la prueba
from pathlib import Path
from shutil import copyfileobj
import json
import os
import subprocess
import sys
from urllib.request import urlopen

REVISION = "e4f670708d0c289489374a10a3fc479e5a39fd5c"
REPOSITORIO = "Rio7u7r/polaridad-textual-robertuito"
PROYECTO = Path("/content/robertuito_aplicativo") / REVISION[:8]

def descargar(archivo):
    destino = PROYECTO / archivo
    if destino.is_file():
        return destino
    servidor = ("https://media.githubusercontent.com/media"
                if destino.suffix == ".safetensors" else "https://raw.githubusercontent.com")
    destino.parent.mkdir(parents=True, exist_ok=True)
    temporal = destino.with_suffix(destino.suffix + ".part")
    url = f"{servidor}/{REPOSITORIO}/{REVISION}/{archivo}"
    with urlopen(url, timeout=60) as respuesta, temporal.open("wb") as salida:
        copyfileobj(respuesta, salida, length=1024 * 1024)
    temporal.replace(destino)
    return destino

for archivo in ("corrida.json", "aplicativo/nucleo.py"):
    descargar(archivo)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.49.0", "huggingface-hub==0.36.2", "pysentimiento==0.7.3",
], check=True)

corrida = json.loads((PROYECTO / "corrida.json").read_text(encoding="utf-8"))["corrida"]

def preparar_modelos(modelos):
    for modelo in modelos:
        carpeta = f"modelos/{corrida}/{modelo}"
        manifiesto = descargar(f"{carpeta}/ejecucion.json")
        registro = json.loads(manifiesto.read_text(encoding="utf-8"))
        for archivo in registro["artefactos_sha256"]:
            descargar(f"{carpeta}/{archivo}")

os.environ["TESIS_VARIANTE"] = "comparativa"
sys.path.insert(0, str(PROYECTO / "aplicativo"))
from nucleo import predecir, validar_texto

print("Preparación completa. Ejecuta la celda 2 para probar un texto.")

In [ ]:
# @title 2. Clasificar un texto
texto = "Me alegra que todo saliera bien."  # @param {type:"string"}
modo = "Solo M2"  # @param ["Solo M2", "Comparar M0, M1 y M2"]

import pandas as pd
from IPython.display import display

validar_texto(texto)
modelos = ("M2",) if modo == "Solo M2" else ("M0", "M1", "M2")
preparar_modelos(modelos)
resultados = []
for modelo in modelos:
    resultado = predecir(texto, modelo)
    probabilidades = resultado["porcentajes"]
    resultados.append({
        "Modelo": modelo,
        "Polaridad": resultado["etiqueta"],
        "Negativa (%)": probabilidades["NEG"],
        "Neutral (%)": probabilidades["NEU"],
        "Positiva (%)": probabilidades["POS"],
    })
display(pd.DataFrame(resultados).round(2))

## Opcional. Abrir la interfaz Gradio

Ejecuta la celda 3 si quieres usar la interfaz completa. Aparece en el
cuaderno y genera un enlace público temporal de Gradio para esa sesión.
Usa textos de ejemplo o textos sin información personal.

In [ ]:
# @title 3. Abrir Gradio (opcional)
abrir_gradio = False  # @param {type:"boolean"}

if abrir_gradio:
    for archivo in ("app.py", "ui_render.py", "ejemplos.py", "ficha_modelos.json"):
        descargar(f"aplicativo/{archivo}")
    requisitos = descargar("requirements-gradio.txt")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requisitos)], check=True)
    preparar_modelos(("M0", "M1", "M2"))

    from starlette.middleware import Middleware
    from app import RecursosLocales, construir_app

    if "demo" in globals():
        demo.close()
    demo = construir_app()
    demo.launch(share=True, inline=True, ssr_mode=False, show_api=False,
                max_file_size=0, blocked_paths=["/content"],
                app_kwargs={"middleware": [Middleware(RecursosLocales)]})